# 10 — Time domain

Analytical Cout/ESR estimates vs ngspice behavioral vs (optional) LTspice vendor model. Load-step recovery time `tau_loop` is an **editable** stand-in; G26 is ~200 µs with 1 µF.

## Setup — editable BOM and run knobs

This cell is shared by every notebook. It puts `P5V_Regulator/` on `sys.path`
and builds **one** `CircuitParams` object. Change a resistor, capacitor, or
tolerance here and re-run: analytical, SPICE, WC, and MC all follow this object.

**Plot callouts:** each figure cell has an `ann = [dict(...), ...]` list.
Edit `text`, `xy` (data coords of the arrow tip), and `offset=(dx, dy)` in
**points** to nudge a label. Use `xytext=(x, y)` instead of `offset` for data
coordinates. `arrow=False` for a plain label. `ann = []` hides all callouts.

Tolerances are **fractions** (0.01 = 1 %). Figures go to `results/figures/`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_project_root() -> Path:
    start = Path.cwd().resolve()
    named = Path("Sections/Electronics/Analog_and_Digital/LT3010_5V_Supply")
    candidates = [start, *start.parents, start / named]
    if start.name == "notebooks":
        candidates.insert(0, start.parent)
    for cand in candidates:
        if (cand / "p5v_design" / "__init__.py").is_file():
            return cand
    return start

ROOT = _find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from p5v_design.params import CircuitParams, ToleranceSpec, format_eng, LdoParams
from p5v_design import analysis, schematic, plots, stability
from p5v_design import plots_wc_mc as pwc
from p5v_design.ldo_model import dropout_v, ignd_a, tj_c, psrr_db, vadj_typical_vs_temp
from dataclasses import replace

# --- EDITABLE BOM ---
# Datasheet TA01, "5V Supply with Shutdown" (LT3010/LT3010-5 Rev. J page 1).
# Fixed 5 V part: SENSE tied to OUT (R1 = 0). No external divider (R2 open).
# Capacitors are the 1 µF parts drawn on that figure. Load is the labeled 50 mA.
# VIN is printed as 5.4 V to 80 V; 12 V is the single-point bias inside that range.
R1 = 0.0             # SENSE shorted to OUT
R2 = 1.0e12          # open — no bottom resistor
CFF = 0.0            # no feedforward capacitor on the fixed part
COUT = 1.0e-6
CIN = 1.0e-6
ESR_OUT = 10e-3      # ceramic assumption, not printed on TA01
ESR_IN = 10e-3
RLOAD = 100.0        # 5 V / 50 mA
R_TOL = 0.01         # 1 %
C_TOL = 0.10         # 10 %
VIN_NOM = 12.0
VIN_MIN = 5.4
VIN_MAX = 80.0
ILOAD_OP = 50e-3
N_MC = 2000
N_SPICE_MC = 40
N_LTSPICE_MC = 200
SKIP_SPICE_WC_MC = False
RUN_LTSPICE_BATCH = False
SPICE_WORKERS = None
LTSPICE_WORKERS = None

p = CircuitParams(
    r1=R1, r2=R2, cff=CFF, cout=COUT, cin=CIN,
    esr_out=ESR_OUT, esr_in=ESR_IN, r_load=RLOAD,
    r_tol=ToleranceSpec(R_TOL), c_tol=ToleranceSpec(C_TOL),
)
p = replace(p, op=replace(p.op, vin_nom=VIN_NOM, vin_min=VIN_MIN, vin_max=VIN_MAX,
                           iload_op=ILOAD_OP, r_load=RLOAD))

FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("gain =", p.gain, "  Vout nom =", analysis.vout_dc(p))
print("SPICE_WORKERS =", SPICE_WORKERS, "  N_MC =", N_MC)


gain = 1.0   Vout nom = 5.0
SPICE_WORKERS = None   N_MC = 2000


In [2]:
from p5v_design.timedomain import load_step, startup, line_step
from p5v_design.simulate import find_ngspice, simulate_tran

TAU_LOOP = 80e-6   # editable recovery time constant
ls = load_step(p, tau_loop=TAU_LOOP)
ann = [
    dict(text="1 → 50 mA", xy=(0.05, ls.vout.min()), offset=(12, -14), arrow=True),
    dict(text="ESR spike", xy=(0.05, ls.vout.min()), offset=(40, 20), arrow=True),
]
plots.plot_tran(ls.t, ls.vout, iout=ls.iout, annotations=ann, path=FIG/"10_load_step.png",
                title="Load step 1→50 mA (analytical)", t_scale=1e3, t_label="t (ms)")
plt.show()
st = startup(p)
ann_s = [dict(text="Ilim charge", xy=(0.2, 2.0), offset=(12, 10), arrow=True)]
plots.plot_tran(st.t, st.vout, vin=st.vin, annotations=ann_s, path=FIG/"10_startup.png",
                title="Startup (Ilim onto Cout)", t_scale=1e3)
plt.show()
ln = line_step(p)
ann_l = [dict(text="9 → 12 V", xy=(0.1, ln.vout[0]), offset=(12, 10), arrow=True)]
plots.plot_tran(ln.t, ln.vout, vin=ln.vin, annotations=ann_l, path=FIG/"10_line_step.png",
                title="Line step", t_scale=1e3)
plt.show()
try:
    find_ngspice()
    sp = simulate_tran(p, kind="load", workdir=ROOT/"results"/"ngspice_nom"/"tran_load")
    plots.plot_tran(ls.t, ls.vout, spice_t=sp.t, spice_v=sp.vout, iout=ls.iout,
                    annotations=[dict(text="ngspice overlay", xy=(0.2, analysis.vout_dc(p)), offset=(10, 10), arrow=True)],
                    path=FIG/"10_load_spice.png", title="Load step vs ngspice")
    plt.show()
except FileNotFoundError as e:
    print(e)
except Exception as e:
    print("ngspice load step failed:", e)
try:
    find_ngspice()
    sp_ln = simulate_tran(p, kind="line", workdir=ROOT/"results"/"ngspice_nom"/"tran_line")
    plots.plot_tran(ln.t, ln.vout, spice_t=sp_ln.t, spice_v=sp_ln.vout, vin=ln.vin,
                    path=FIG/"10_line_spice.png", title="Line step vs ngspice")
    plt.show()
    sp_st = simulate_tran(p, kind="startup", workdir=ROOT/"results"/"ngspice_nom"/"tran_start")
    plots.plot_tran(st.t, st.vout, spice_t=sp_st.t, spice_v=sp_st.vout, vin=st.vin,
                    path=FIG/"10_startup_spice.png", title="Startup vs ngspice")
    plt.show()
except Exception as e:
    print("ngspice extra transients:", e)
dip = float(analysis.vout_dc(p) - ls.vout.min())
print(f"analytical load-step undershoot {1e3*dip:.1f} mV")
fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.barh(["undershoot (mV)", "ESR spike (mV)", "Cout dip (mV)"],
        [1e3*dip, 1e3*(p.op.iload_op-1e-3)*p.esr_out, 1e3*(p.op.iload_op-1e-3)*80e-6/p.cout],
        color=["C3", "C0", "C1"], edgecolor="k")
ax.set_xlabel("mV")
ax.set_title("Load-step budget (analytical)")
ax.grid(True, axis="x", alpha=0.3)
plots.savefig(fig, FIG/"10_step_budget.png"); plt.show()
# Load-step family (HV_monitor 01 analog)
fam = {}
for i1, lab in ((10e-3, "1→10 mA"), (50e-3, "1→50 mA"), (50e-3, "1→50 mA")):
    tr = load_step(p, i0=1e-3, i1=i1, tau_loop=TAU_LOOP)
    fam[lab] = (tr.t, tr.vout)
plots.plot_tran_family(fam, title="Load-step family (analytical)", path=FIG/"10_load_family.png")
plt.show()
fall = load_step(p, i0=p.op.iload_op, i1=1e-3, tau_loop=TAU_LOOP)
plots.plot_tran(fall.t, fall.vout, iout=fall.iout,
                path=FIG/"10_load_fall.png", title="Load step 35→1 mA (analytical)",
                t_scale=1e3, t_label="t (ms)")
plt.show()
esr_fam = {}
for esr, lab in ((1e-3, "1 mΩ"), (5e-3, "5 mΩ BOM"), (50e-3, "50 mΩ"), (300e-3, "300 mΩ max")):
    pc = replace(p, esr_out=esr)
    tr = load_step(pc, tau_loop=TAU_LOOP)
    esr_fam[lab] = (tr.t, tr.vout)
plots.plot_tran_family(esr_fam, title="Load step vs Cout ESR", path=FIG/"10_esr_family.png")
plt.show()
c_fam = {}
for c, lab in ((1e-6, "1 µF"), (3.3e-6, "3.3 µF"), (10e-6, "10 µF BOM"), (22e-6, "22 µF")):
    pc = replace(p, cout=c)
    tr = load_step(pc, tau_loop=TAU_LOOP)
    c_fam[lab] = (tr.t, tr.vout)
plots.plot_tran_family(c_fam, title="Load step vs Cout", path=FIG/"10_cout_family.png")
plt.show()

cell:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


...

analytical load-step undershoot 3919.8 mV


cell:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:81: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
